# Accident Detection — Temporal Attention-MIL Model

<div dir="rtl">

این نوت‌بوک مدل اصلی تشخیص تصادف را می‌سازد و ارزیابی می‌کند. ورودی، خروجی نوت‌بوک پری‌پراسس است: برای هر ویدئو یک دنباله‌ی زمانی از بردارهای CLIP، DINOv2 و حرکت، هر ۰٫۲ ثانیه یک بردار، از ابتدا تا انتهای ویدئو.

**ایده:** هر ویدئو یک «کیسه» از لحظه‌هاست و فقط برچسب کل ویدئو معلوم است، نه اینکه حادثه در کدام لحظه رخ داده. به این حالت Multiple Instance Learning (MIL) می‌گویند. مدل برای هر لحظه با نگاه به حدود ۲ ثانیه‌ی اطرافش یک نمایش می‌سازد و با attention یاد می‌گیرد کدام لحظه‌ها مهم‌اند؛ امتیاز ویدئو از همین لحظه‌ها ساخته می‌شود.

**قواعد:**
- `time_of_event` نه در انتخاب فریم، نه در ورودی و نه در آموزش استفاده می‌شود. فقط در بخش ۱۲، بعد از آموزش، خوانده می‌شود تا ببینیم مدل به کجای ویدئو نگاه کرده است.
- همه‌ی تصمیم‌ها (معماری، ویژگی‌ها، تعداد epoch) با 5-fold CV روی ۱۲۰۰ ویدئوی dev گرفته می‌شوند. ۳۰۰ ویدئوی test در این نوت‌بوک استفاده نمی‌شوند و فقط یک بار، در ارزیابی نهایی، به کار می‌روند.
- آماره‌های نرمال‌سازی در هر fold فقط از ویدئوهای train همان fold حساب می‌شوند.

</div>

## 1. Setup

In [1]:
import json
import math
import re
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss, confusion_matrix
from sklearn.model_selection import GridSearchCV, PredefinedSplit, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/accident_detection")
CACHE_PATH = PROJECT_DIR / "cache_train.npz"
RUN_DIR = PROJECT_DIR / "mil_runs"      # saved CV results; delete a file to rerun that model
RUN_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu")

Mounted at /content/drive
Device: Tesla T4


## 2. Feature Cache

<div dir="rtl">

خواندن ۱۵۰۰ فایل کوچک از Drive کند است، چون هر فایل جدا تأخیر دارد. فقط بار اول همه‌ی فایل‌ها با چند thread خوانده و پشت سر هم در یک فایل `cache_train.npz` (حدود ۱٫۲ گیگابایت) ذخیره می‌شوند؛ اجراهای بعدی فقط همین یک فایل را می‌خوانند. ویدئوهای test هم در cache هستند، ولی در این نوت‌بوک فقط ردیف‌های dev برداشته می‌شوند.

</div>

In [2]:
splits = pd.read_csv(PROJECT_DIR / "splits.csv", dtype={"video_id": str})


def load_features(video_id):
    with np.load(PROJECT_DIR / "features" / "train" / f"{video_id}.npz") as f:
        return f["clip"], f["dino"], f["motion"], f["times"]


if not CACHE_PATH.exists():
    started = time.time()
    with ThreadPoolExecutor(max_workers=8) as pool:
        videos = list(tqdm(pool.map(load_features, splits["video_id"]), total=len(splits)))
    clip, dino, motion, times = (np.concatenate(part) for part in zip(*videos))
    tmp = CACHE_PATH.with_suffix(".tmp.npz")
    np.savez(tmp, video_id=splits["video_id"].to_numpy(str), lengths=[len(v[3]) for v in videos],
             clip=clip, dino=dino, motion=motion, times=times)
    tmp.rename(CACHE_PATH)    # a half-written cache never gets the final name
    del videos, clip, dino, motion, times
    print(f"Cache built in {time.time() - started:.0f} s")

started = time.time()
cache = dict(np.load(CACHE_PATH))
assert (cache["video_id"] == splits["video_id"].to_numpy(str)).all()    # same order as splits.csv
offsets = np.r_[0, np.cumsum(cache["lengths"])]
print(f"{len(splits)} videos, {offsets[-1]:,} moments, loaded in {time.time() - started:.0f} s "
      f"({CACHE_PATH.stat().st_size / 1e9:.2f} GB)")

1500 videos, 283,387 moments, loaded in 33 s (1.17 GB)


## 3. Model Inputs

<div dir="rtl">

ورودی هر لحظه (هر ۰٫۲ ثانیه) یک بردار ۲۰۶۳ بعدی است:

| بخش | ابعاد | نقش |
|---|---|---|
| `clip` | ۵۱۲ | معنای صحنه: خودرو، عابر، تقاطع |
| `dino` | ۱۵۳۶ | جزئیات مکانی و شکل اجسام (CLS + میانگین patchها) |
| `motion` | ۶ | شدت حرکت، سرعت تقریبی، نزدیک شدن، انحراف، تکان |
| `d_motion` | ۶ | تغییر همان ۶ عدد نسبت به لحظه‌ی قبل؛ ترمز ناگهانی یعنی افت تند `flow_bottom` |
| `change` | ۳ | فاصله‌ی cosine هر embedding با لحظه‌ی قبل (CLIP، DINO CLS، DINO patch)؛ تغییر ناگهانی کل تصویر |

- **تفاوت با لحظه‌ی قبل:** برای حرکت، خود تفاوت داده می‌شود. برای embeddingها تفاوت کامل ۲۰۴۸ بعدی ورودی را دو برابر می‌کند و با ۹۶۰ ویدئوی آموزشی overfitting را بیشتر می‌کند؛ ضمن اینکه لایه‌ی temporal conv مدل (بخش ۴) خودش تفاوت لحظه‌های پشت سر هم را یاد می‌گیرد. چیزی که conv به‌سادگی نمی‌سازد «چقدر کل تصویر عوض شد» است و آن را با سه عدد cosine صریح می‌دهیم.
- **تبدیل log برای حرکت:** مقادیر optical flow دم سنگین دارند و چند لحظه‌ی ترمز شدید بسیار بزرگ‌اند. بدون تبدیل، همین چند مقدار میانگین و انحراف معیار نرمال‌سازی را تعیین می‌کنند. `sign(x)·log(1+|x|)` جهت حرکت را نگه می‌دارد و دامنه را فشرده می‌کند.
- **نرمال‌سازی:** z-score برای هر ستون، با آماره‌های لحظه‌های ویدئوهای train همان fold؛ آماره‌ها در بخش ۵ حساب و داخل خود مدل اعمال می‌شوند. آماره‌ی کل dev اطلاعات ویدئوهای validation را وارد آموزش می‌کند.
- **طول متغیر:** ویدئوها ۷۵ تا ۳۰۰ لحظه دارند. همه تا بلندترین ویدئو با صفر پر می‌شوند و یک mask مشخص می‌کند کدام لحظه واقعی است؛ مدل لحظه‌های پرشده را نمی‌بیند. برش یا نمونه‌برداری به طول ثابت لازم نیست و هیچ بخشی از ویدئو دور ریخته نمی‌شود.

کل ورودی ۱۲۰۰ ویدئوی dev حدود ۱٫۵ گیگابایت است و یک بار روی GPU می‌رود؛ آموزش دیگر منتظر خواندن داده نمی‌ماند.

</div>

In [3]:
COLS = {"clip": (0, 512), "dino": (512, 2048), "motion": (2048, 2054), "d_motion": (2054, 2060),
        "clip_change": (2060, 2061), "dino_change": (2061, 2063)}
N_FEATURES = 2063


def columns(*parts):
    return np.concatenate([np.arange(*COLS[p]) for p in parts])


FEATURE_SETS = {
    "All": columns("clip", "dino", "motion", "d_motion", "clip_change", "dino_change"),
    "No change features": columns("clip", "dino", "motion"),
    "CLIP": columns("clip", "clip_change"),
    "DINOv2": columns("dino", "dino_change"),
    "Motion": columns("motion", "d_motion"),
    "Motion + change": columns("motion", "d_motion", "clip_change", "dino_change"),
}


def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))


def cosine_change(x):
    x = x / np.linalg.norm(x, axis=1, keepdims=True)
    return np.r_[0, 1 - (x[1:] * x[:-1]).sum(1)]


def moment_features(row):
    s, e = offsets[row], offsets[row + 1]
    clip = cache["clip"][s:e].astype(np.float32)
    dino = cache["dino"][s:e].astype(np.float32)
    motion = signed_log(cache["motion"][s:e])
    d_motion = np.diff(motion, axis=0, prepend=motion[:1])
    change = np.stack([cosine_change(clip), cosine_change(dino[:, :768]), cosine_change(dino[:, 768:])], axis=1)
    return np.hstack([clip, dino, motion, d_motion, change])


dev_rows = np.flatnonzero(splits["fold"] >= 0)
dev = splits.iloc[dev_rows].reset_index(drop=True)
lengths = cache["lengths"][dev_rows]
X = np.zeros((len(dev), lengths.max(), N_FEATURES), np.float16)
for j, row in enumerate(tqdm(dev_rows)):
    X[j, :lengths[j]] = moment_features(row)

X = torch.from_numpy(X).to(device)
lengths_t = torch.as_tensor(lengths, device=device)
mask = torch.arange(X.shape[1], device=device) < lengths_t[:, None]
y = torch.tensor(dev["label"].to_numpy(), dtype=torch.float32, device=device)
y_np, fold = dev["label"].to_numpy(), dev["fold"].to_numpy()

print("X:", tuple(X.shape), f"| {X.numel() * 2 / 1e9:.2f} GB on {device} | finite: {bool(torch.isfinite(X).all())}")
display(pd.crosstab(dev["fold"], dev["label"]))

  0%|          | 0/1200 [00:00<?, ?it/s]

X: (1200, 300, 2063) | 1.49 GB on cuda | finite: True


label,0,1
fold,,
0,121,119
1,120,120
2,120,120
3,119,121
4,119,121


## 4. Model

<div dir="rtl">

```
moments (T × 2063) → Linear 128 → [LayerNorm → Conv1d k=5 → GELU] × 2 (residual) → gated attention → Linear → logit
```

1. **embedding لحظه:** یک Linear از ابعاد ورودی به ۱۲۸ بعد، مشترک برای همه‌ی لحظه‌ها.
2. **temporal conv:** دو لایه‌ی Conv1d با kernel 5. هر لحظه بعد از دو لایه ۹ لحظه، یعنی ۱٫۸ ثانیه، را می‌بیند.
3. **gated attention pooling:** برای هر لحظه یک وزن (جمع وزن‌ها ۱)؛ نمایش ویدئو میانگین وزن‌دار لحظه‌هاست.
4. **head:** یک Linear که احتمال تصادف را می‌دهد.

**چرا این انتخاب‌ها:**
- **ویژگی‌های frozen، نه fine-tune:** با ۹۶۰ ویدئوی آموزشی در هر fold، fine-tune کردن یک encoder بزرگ (مثل MViT با حدود ۳۵ میلیون پارامتر) سریع overfit می‌شود و روی T4 ساعت‌ها طول می‌کشد. اینجا حداکثر حدود ۴۵۰ هزار پارامتر آموزش می‌بیند و هر fold چند ثانیه طول می‌کشد.
- **MIL، نه برچسب برای هر فریم:** برچسب فقط در سطح ویدئوست. اگر همه‌ی فریم‌های ویدئوی تصادف برچسب ۱ بگیرند، بیش از ۹۰٪ آن‌ها رانندگی عادی‌اند و مدل نویز یاد می‌گیرد؛ ساختن برچسب فریم از `time_of_event` هم ممنوع است.
- **temporal conv کوتاه، نه LSTM یا Transformer:** حادثه ۱ تا ۲ ثانیه طول می‌کشد و conv همین الگوی محلی (ترمز ناگهانی، نزدیک شدن سریع) را می‌بیند. conv به جای مطلق حساس نیست: یک الگو در ثانیه‌ی ۵ و ثانیه‌ی ۲۰ یک خروجی دارد. LSTM و GRU می‌توانند گام‌ها را از ابتدای ویدئو بشمارند و «ثانیه‌ی ۲۰» را یاد بگیرند. Transformer هم بدون positional encoding ترتیب لحظه‌ها را نمی‌بیند و با positional encoding جایگاه مطلق را هم می‌بیند. conv همچنین پارامتر کمتری دارد و موازی اجرا می‌شود.
- **بدون positional encoding:** ۸۷٪ حادثه‌ها بین ثانیه‌ی ۱۸ و ۲۲ هستند. مدلی که جایگاه زمانی را بداند می‌تواند به‌جای دیدن حادثه فقط به ثانیه‌ی ۲۰ نگاه کند؛ این در استفاده‌ی واقعی کار نمی‌کند. conv و attention pooling هر دو به جابه‌جایی زمانی حساس نیستند.
- **attention، نه میانگین یا max:** میانگین لحظه‌ی کوتاه حادثه را در ۴۰ ثانیه رانندگی عادی رقیق می‌کند؛ همین مشکل linear probe بود. max فقط به یک لحظه تکیه می‌کند و به یک فریم پرت حساس است. attention وزن‌ها را یاد می‌گیرد و می‌تواند روی چند لحظه‌ی مهم تمرکز کند. نسخه‌ی gated (Ilse و همکاران، ۲۰۱۸) یک دروازه‌ی sigmoid را در tanh ضرب می‌کند و وزن‌های دقیق‌تری از tanh تنها می‌دهد. میانگین و max در بخش ۷ امتحان می‌شوند.
- **regularization:** dropout روی ورودی و لایه‌ها (۳۰٪)، weight decay، و time drop: در آموزش ۱۰٪ لحظه‌ها تصادفی حذف می‌شوند تا مدل به یک فریم تکی وابسته نشود. این حذف تصادفی است و به زمان حادثه ربطی ندارد. لحظه‌های حذف‌شده و پرشده بعد از هر لایه دوباره صفر می‌شوند تا conv و attention آن‌ها را نبینند.

سه گزینه‌ی دیگر مدل (`center`، `pca` و `edge_trim`) جواب مشکلی هستند که در بخش ۶ دیده می‌شود و در بخش ۸ توضیح داده می‌شوند. پیش‌فرض هر سه خاموش است.

</div>

In [4]:
class AttentionMIL(nn.Module):
    def __init__(self, cols, mean, std, dim=128, kernel=5, layers=2, dropout=0.3, input_dropout=0.3,
                 time_drop=0.1, pooling="attention", center=False, basis=None, edge_trim=False):
        super().__init__()
        cols = torch.as_tensor(cols, device=mean.device)
        self.register_buffer("cols", cols)
        self.register_buffer("mean", mean[cols])     # train-fold statistics
        self.register_buffer("std", std[cols])
        self.register_buffer("basis", basis)         # PCA directions of the appearance columns, or None
        self.n_app = int((cols < 2048).sum())        # appearance (CLIP, DINOv2) columns come first
        self.center, self.time_drop, self.pooling = center, time_drop, pooling
        self.edge = layers * (kernel // 2) + 1 if edge_trim else 0
        n_in = len(cols) - self.n_app + (basis.shape[1] if basis is not None else self.n_app)
        self.embed = nn.Sequential(nn.Dropout(input_dropout), nn.Linear(n_in, dim), nn.GELU(), nn.Dropout(dropout))
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(layers)])
        self.convs = nn.ModuleList([nn.Conv1d(dim, dim, kernel, padding=kernel // 2) for _ in range(layers)])
        self.drop = nn.Dropout(dropout)
        self.att_v, self.att_u, self.att_w = nn.Linear(dim, 64), nn.Linear(dim, 64), nn.Linear(64, 1)
        self.head = nn.Linear(dim, 1)

    def forward(self, x, mask):
        m = mask.unsqueeze(-1).float()
        x = (x[..., self.cols].float() - self.mean) / self.std
        if self.center or self.basis is not None:
            app, rest = x[..., :self.n_app], x[..., self.n_app:]
            if self.center:                          # keep only what changes inside this video
                app = app - (app * m).sum(1, keepdim=True) / m.sum(1, keepdim=True)
            if self.basis is not None:
                app = app @ self.basis
            x = torch.cat([app, rest], dim=-1)

        pool = mask                                  # moments that take part in pooling
        if self.edge:                                # only moments whose whole conv window is inside the video
            pos = torch.arange(mask.shape[1], device=mask.device)
            pool = mask & (pos >= self.edge) & (pos < mask.sum(1, keepdim=True) - self.edge)
        if self.training and self.time_drop:
            keep = torch.rand(mask.shape, device=mask.device) > self.time_drop
            mask, pool = mask & keep, pool & keep
            m = mask.unsqueeze(-1).float()

        h = self.embed(x) * m
        for norm, conv in zip(self.norms, self.convs):
            u = (norm(h) * m).transpose(1, 2)        # conv works on (batch, channels, time)
            h = (h + self.drop(F.gelu(conv(u))).transpose(1, 2)) * m

        if self.pooling == "max":
            moment_score = self.head(h).squeeze(-1).masked_fill(~pool, float("-inf"))   # for inspection only
            return self.head(h.masked_fill(~pool.unsqueeze(-1), -1e4).max(1).values).squeeze(-1), moment_score
        if self.pooling == "mean":
            score = torch.zeros_like(m.squeeze(-1))
        else:
            score = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).squeeze(-1)
        weights = score.masked_fill(~pool, float("-inf")).softmax(1)
        video = (weights.unsqueeze(-1) * h).sum(1)
        return self.head(video).squeeze(-1), weights


check = AttentionMIL(FEATURE_SETS["All"], X.new_zeros(N_FEATURES, dtype=torch.float32),
                     X.new_ones(N_FEATURES, dtype=torch.float32)).to(device).eval()
logit, weights = check(X[:4], mask[:4])
print(f"Parameters: {sum(p.numel() for p in check.parameters()):,}")
print("Output:", tuple(logit.shape), "| attention:", tuple(weights.shape),
      "| weights sum to 1:", bool(torch.allclose(weights.sum(1), torch.ones(4, device=device))),
      "| zero weight on padding:", bool((weights[~mask[:4]] == 0).all()))

Parameters: 445,506
Output: (4,) | attention: (4, 300) | weights sum to 1: True | zero weight on padding: True


## 5. Training and Cross-Validation

<div dir="rtl">

- **loss و optimizer:** binary cross-entropy و AdamW (lr = 3e-4، weight decay = 0.05)، batch 32 ویدئو.
- **lr ثابت:** با lr ثابت، مدل بعد از epoch شماره‌ی e همان مدلی است که یک اجرای e‌ epochی می‌ساخت. پس یک اجرای ۴۰ epochی کل منحنی را می‌دهد و تعداد epoch بدون اجرای دوباره انتخاب می‌شود. با یک schedule مثل cosine، هر تعداد epoch یک اجرای جدا لازم داشت.
- **انتخاب تعداد epoch:** epochی که log loss روی ۱۲۰۰ پیش‌بینی out-of-fold کمترین است، برای همه‌ی foldها یکسان. انتخاب بهترین epoch برای هر fold جدا، و انتخاب بر اساس accuracy (که با یک ویدئو بالا و پایین می‌رود)، هر دو عدد را خوش‌بینانه می‌کنند.
- **۳ seed در هر fold:** با ۹۶۰ ویدئو نتیجه به مقداردهی اولیه حساس است. احتمال‌های سه مدل میانگین گرفته می‌شود؛ نوسان کم می‌شود و در ارزیابی نهایی هم همین ۱۵ مدل (۵ fold × ۳ seed) روی test میانگین گرفته می‌شوند.
- **ارزیابی:** هر ویدئوی dev دقیقاً یک بار در validation است؛ accuracy روی ۱۲۰۰ پیش‌بینی out-of-fold حساب می‌شود. بازه‌ی اطمینان ۹۵٪ (Wilson) برای ۱۲۰۰ ویدئو حدود ۲٫۵± درصد است؛ برای ۶۰ ویدئو حدود ۱۱± درصد.
- **آستانه‌ی ۰٫۵:** کلاس‌ها متوازن‌اند. آستانه‌ای که روی validation تنظیم شود عدد همان validation را خوش‌بینانه می‌کند.
- **ذخیره‌ی نتایج:** نتیجه‌ی CV هر مدل در `mil_runs` ذخیره می‌شود؛ اگر اتصال قطع شد، مدل‌های تمام‌شده دوباره آموزش نمی‌بینند.
- **تنظیمات هر مدل:** هر config می‌تواند lr، weight decay، سقف epoch و mirror augmentation خودش را داشته باشد (بخش‌های ۹ و ۱۰)؛ اگر نداشته باشد، مقدارهای پیش‌فرض بالا استفاده می‌شوند.

</div>

In [5]:
EPOCHS = 40          # upper limit; the number used is chosen from the validation curve
BATCH = 32
LR = 3e-4
WEIGHT_DECAY = 0.05
SEEDS = [0, 1, 2]
EPOCHS_LONG = 80     # for the models of sections 9 and 10
SEEDS_FINAL = [10, 11, 12]
APP = slice(0, 2048)  # appearance columns: CLIP + DINOv2

# mirror (left <-> right) of the dynamic columns: only the horizontal motion changes sign
FLIP_SIGN = np.ones(N_FEATURES)
FLIP_SIGN[[COLS["motion"][0] + 3, COLS["d_motion"][0] + 3]] = -1     # flow_x and its change
FLIP_SIGN = torch.as_tensor(FLIP_SIGN, dtype=X.dtype, device=device)


def frame_stats(idx):
    """Mean and std of every input column over the real moments of the given videos."""
    total = torch.zeros(N_FEATURES, dtype=torch.float64, device=device)
    total_sq = torch.zeros_like(total)
    count = 0
    for b in torch.as_tensor(idx, device=device).split(64):
        x = X[b][mask[b]].double()
        total += x.sum(0)
        total_sq += (x ** 2).sum(0)
        count += len(x)
    mean = total / count
    std = (total_sq / count - mean ** 2).clamp_min(0).sqrt()
    return mean.float(), std.float().clamp_min(1e-4)


def pca_basis(idx, mean, std, k, center):
    """Top-k principal directions of the normalized appearance columns of the given videos."""
    cov = torch.zeros(2048, 2048, dtype=torch.float64, device=device)
    count = 0
    for b in torch.as_tensor(idx, device=device).split(32):
        x = (X[b, :, APP].float() - mean[APP]) / std[APP]
        m = mask[b].unsqueeze(-1).float()
        if center:
            x = x - (x * m).sum(1, keepdim=True) / m.sum(1, keepdim=True)
        x = x[mask[b]]
        cov += (x.T @ x).double()
        count += len(x)
    _, vectors = torch.linalg.eigh(cov / count)      # eigenvalues in ascending order
    return vectors[:, -k:].float()


@torch.no_grad()
def predict(model, idx, with_weights=False, data=None):
    """Probabilities of the given videos; `data` = (X, mask, lengths) of another set, default dev."""
    Xd, mask_d, lengths_d = data or (X, mask, lengths_t)
    model.eval()
    probs, weights = [], []
    for b in torch.as_tensor(idx, device=device).split(128):
        L = int(lengths_d[b].max())
        logit, w = model(Xd[b, :L], mask_d[b, :L])
        probs.append(torch.sigmoid(logit).cpu())
        if with_weights:
            weights += [wi[:n].cpu().numpy() for wi, n in zip(w, lengths_d[b].tolist())]
    probs = torch.cat(probs).numpy()
    return (probs, weights) if with_weights else probs


def train(cfg, train_idx, seed, epochs, watch=None):
    """Train one model. After every epoch, predict the videos in `watch` (name -> indices)."""
    torch.manual_seed(seed)
    mean, std = frame_stats(train_idx)               # train videos of this fold only
    args = dict(cfg["model"])
    k = args.pop("pca", None)
    basis = pca_basis(train_idx, mean, std, k, args.get("center", False)) if k else None
    model = AttentionMIL(FEATURE_SETS[cfg["features"]], mean, std, basis=basis, **args).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.get("lr", LR), weight_decay=cfg.get("weight_decay", WEIGHT_DECAY))
    flip = cfg.get("flip", False)
    assert not (flip and model.n_app), "CLIP/DINOv2 embeddings cannot be mirrored"
    train_idx = torch.as_tensor(train_idx, device=device)
    history = {name: [] for name in (watch or {})}
    for _ in range(epochs):
        model.train()
        for b in train_idx[torch.randperm(len(train_idx), device=device)].split(BATCH):
            L = int(lengths_t[b].max())
            x = X[b, :L]
            if flip:                                     # mirror half of the videos
                mirrored = torch.rand(len(b), device=device) < 0.5
                x = torch.where(mirrored[:, None, None], x * FLIP_SIGN, x)
            logit, _ = model(x, mask[b, :L])
            loss = F.binary_cross_entropy_with_logits(logit, y[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
        for name, idx in (watch or {}).items():
            history[name].append(predict(model, idx))
    return model, history


def run_cv(name, cfg, curves=False, seeds=SEEDS):
    """5-fold CV, several seeds per fold. Returns the out-of-fold probability of every dev video after every epoch."""
    path = RUN_DIR / f"{name}.npz"
    if path.exists():
        return dict(np.load(path))
    epochs = cfg.get("epochs", EPOCHS)
    out = {"oof": np.zeros((epochs, len(dev)))}
    if curves:
        out["train_loss"], out["train_acc"] = np.zeros((epochs, 5)), np.zeros((epochs, 5))
    for k in tqdm(range(5), desc=name):
        tr, va = np.flatnonzero(fold != k), np.flatnonzero(fold == k)
        watch = {"val": va, "train": tr} if curves else {"val": va}
        runs = [train(cfg, tr, seed, epochs, watch)[1] for seed in seeds]
        out["oof"][:, va] = np.mean([r["val"] for r in runs], axis=0)     # seed average
        if curves:
            p_train = np.mean([r["train"] for r in runs], axis=0)
            out["train_loss"][:, k] = [log_loss(y_np[tr], p) for p in p_train]
            out["train_acc"][:, k] = [accuracy_score(y_np[tr], p >= 0.5) for p in p_train]
    np.savez(path, **out)
    return out


def metrics(label, p, epochs=None):
    pred = p >= 0.5
    fold_acc = [accuracy_score(y_np[fold == k], pred[fold == k]) for k in range(5)]
    return {"model": label, "epochs": epochs, "accuracy": accuracy_score(y_np, pred),
            "fold_acc_std": np.std(fold_acc), "f1": f1_score(y_np, pred),
            "roc_auc": roc_auc_score(y_np, p), "log_loss": log_loss(y_np, p)}


def summarize(label, oof):
    """Metrics at the epoch with the lowest out-of-fold log loss."""
    best = int(np.argmin([log_loss(y_np, p) for p in oof]))
    return metrics(label, oof[best], best + 1)


def wilson(acc, n, z=1.96):
    """95% confidence interval of an accuracy measured on n videos."""
    acc, n = np.asarray(acc, float), np.asarray(n, float)
    center = (acc + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(acc * (1 - acc) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return center - half, center + half


def metric_bars(table, title):
    """Accuracy (with 95% CI), F1 and ROC-AUC side by side; the model with the lowest log loss in red."""
    acc = table["accuracy"].to_numpy()
    lo, hi = wilson(acc, len(dev))
    colors = np.where(table["log_loss"] == table["log_loss"].min(), "#d62728", "#1f77b4")
    fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                        subplot_titles=["Accuracy (95% CI)", "F1", "ROC-AUC"])
    for col, metric in enumerate(["accuracy", "f1", "roc_auc"], start=1):
        values = table[metric].to_numpy()
        error = dict(type="data", symmetric=False, array=hi - acc, arrayminus=acc - lo) if metric == "accuracy" else None
        fig.add_trace(go.Bar(x=values, y=table["model"], orientation="h", marker_color=colors, text=values.round(3),
                             error_x=error, showlegend=False), row=1, col=col)
    fig.update_xaxes(range=[0.5, 0.9])
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title=title, height=45 * len(table) + 150)
    fig.show()


CONFIGS, OOF = {}, {}     # every model tried: label -> config, label -> out-of-fold probabilities per epoch

## 6. First Model: All Features

<div dir="rtl">

مدل اول با همه‌ی ویژگی‌ها و تنظیمات بخش ۴. منحنی train در حالت eval (بدون dropout) روی ویدئوهای train همان fold حساب می‌شود تا با validation قابل مقایسه باشد. فاصله‌ی دو منحنی میزان overfitting را نشان می‌دهد و خط‌چین epoch انتخاب‌شده است.

</div>

In [6]:
MAIN = {"features": "All", "model": {}}     # AttentionMIL defaults
cv_main = run_cv("main", MAIN, curves=True)
CONFIGS["Attention-MIL (main)"], OOF["Attention-MIL (main)"] = MAIN, cv_main["oof"]
main = summarize("Attention-MIL (main)", cv_main["oof"])
main_epochs = main["epochs"]

display(pd.DataFrame([main]).round(3))
lo, hi = wilson(main["accuracy"], len(dev))
print(f"Accuracy {main['accuracy']:.3f} (95% CI {lo:.3f}-{hi:.3f}) at epoch {main_epochs}")

,model,epochs,accuracy,fold_acc_std,f1,roc_auc,log_loss
0,Attention-MIL (main),1,0.688,0.008,0.67,0.751,0.597


Accuracy 0.688 (95% CI 0.662-0.714) at epoch 1


In [7]:
oof = cv_main["oof"]
epochs = np.arange(1, EPOCHS + 1)
val_loss = np.array([[log_loss(y_np[fold == k], p[fold == k]) for k in range(5)] for p in oof])
val_acc = np.array([[accuracy_score(y_np[fold == k], p[fold == k] >= 0.5) for k in range(5)] for p in oof])

fig = make_subplots(rows=1, cols=2, subplot_titles=["Log loss", "Accuracy"])
for col, (train_m, val_m) in enumerate([(cv_main["train_loss"], val_loss), (cv_main["train_acc"], val_acc)], start=1):
    for name, values, color in [("train", train_m, "#7f7f7f"), ("validation", val_m, "#d62728")]:
        mean, std = values.mean(1), values.std(1)
        fig.add_trace(go.Scatter(x=np.r_[epochs, epochs[::-1]], y=np.r_[mean + std, (mean - std)[::-1]],
                                 fill="toself", fillcolor=color, opacity=0.15, line_width=0, mode="lines",
                                 showlegend=False, hoverinfo="skip"), row=1, col=col)
        fig.add_trace(go.Scatter(x=epochs, y=mean, name=name, line_color=color, showlegend=col == 1), row=1, col=col)
fig.add_vline(x=main_epochs, line_dash="dash", line_color="black")
fig.update_xaxes(title_text="epoch")
fig.update_layout(height=420, title="Training curves of the first model (mean ± std over 5 folds)")
fig.show()

<div dir="rtl">

**نتیجه:** کمترین log loss در epoch 1 است (accuracy ۰٫۶۸۸، AUC ۰٫۷۵۱). از epoch 2 به بعد train accuracy از ۷۵٪ به ۹۹٪ (epoch 9) می‌رسد و validation loss از ۰٫۶۰ به بیش از ۲ بالا می‌رود، در حالی که validation accuracy بین ۶۶ و ۶۹٪ ثابت می‌ماند. مدل ویدئوهای train را حفظ می‌کند، به‌جای اینکه الگوی حادثه را یاد بگیرد.

توضیح: ۲۰۴۸ بعد ویژگی ظاهری در هر لحظه، برای هر ویدئو یک «اثر انگشت» می‌سازد (نوع جاده، نور، ماشین جلویی) که در کل ویدئو تقریباً ثابت است. با ۹۶۰ ویدئو، حفظ کردن این اثر انگشت‌ها از یاد گرفتن الگوی کوتاه حادثه آسان‌تر است. عدد ۰٫۶۸۸ حتی از linear probe (۰٫۶۹۹) پایین‌تر است. بخش ۷ نشان می‌دهد مشکل از ورودی است یا از معماری، و بخش ۸ این توضیح را آزمایش می‌کند.

</div>

## 7. Ablations

<div dir="rtl">

هر ردیف فقط یک چیز را نسبت به مدل اول عوض می‌کند و بقیه (foldها، seedها، قاعده‌ی انتخاب epoch) یکسان است:

| ردیف | سؤال |
|---|---|
| Mean pooling / Max pooling | آیا attention از میانگین و max بهتر است؟ |
| No temporal conv | آیا دیدن ۱٫۸ ثانیه‌ی اطراف هر لحظه کمک می‌کند؟ |
| Longer context (3.4 s) | پنجره‌ی بلندتر (kernel 9) بهتر است یا کوتاه کافی است؟ |
| No change features | آیا `d_motion` و `change` (تفاوت با لحظه‌ی قبل) کمک می‌کنند؟ |
| CLIP / DINOv2 / Motion only | سهم هر نوع ویژگی چقدر است؟ |

</div>

In [8]:
ABLATIONS = {
    "mean_pool": ("Mean pooling", {"model": {"pooling": "mean"}}),
    "max_pool": ("Max pooling", {"model": {"pooling": "max"}}),
    "no_conv": ("No temporal conv", {"model": {"layers": 0}}),
    "kernel9": ("Longer context (3.4 s)", {"model": {"kernel": 9}}),
    "no_change": ("No change features", {"features": "No change features"}),
    "clip": ("CLIP only", {"features": "CLIP"}),
    "dino": ("DINOv2 only", {"features": "DINOv2"}),
    "motion": ("Motion only", {"features": "Motion"}),
}

rows = [main]
for key, (label, change) in ABLATIONS.items():
    cfg = {"features": change.get("features", "All"), "model": change.get("model", {})}
    CONFIGS[label], OOF[label] = cfg, run_cv(key, cfg)["oof"]
    rows.append(summarize(label, OOF[label]))
results = pd.DataFrame(rows)
display(results.round(3))
metric_bars(results, "Ablations of the first model (dev, 5-fold CV)")

,model,epochs,accuracy,fold_acc_std,f1,roc_auc,log_loss
0,Attention-MIL (main),1,0.688,0.008,0.670,0.751,0.597
1,Mean pooling,1,0.688,0.007,0.669,0.750,0.598
2,Max pooling,3,0.707,0.012,0.689,0.773,0.572
3,No temporal conv,3,0.690,0.017,0.686,0.765,0.580
4,Longer context (3.4 s),1,0.691,0.014,0.678,0.750,0.595
5,No change features,1,0.686,0.012,0.670,0.746,0.599
6,CLIP only,1,0.684,0.013,0.675,0.745,0.601
7,DINOv2 only,1,0.683,0.005,0.666,0.741,0.610
8,Motion only,21,0.732,0.035,0.730,0.810,0.543


<div dir="rtl">

**نتیجه:**
- **Motion only بهترین است:** accuracy ۰٫۷۳۲ (بازه‌ی ۰٫۷۰۷ تا ۰٫۷۵۶)، AUC ۰٫۸۱۰، و بهترین epoch ۲۱ است، نه ۱. با ۱۲ عدد در هر لحظه چیزی برای حفظ کردن نیست و مدل باید الگوی زمانی حرکت را یاد بگیرد. همین ویژگی‌ها در linear probe (میانگین در زمان) ۰٫۶۸۲ گرفتند؛ اختلاف ۵ درصدی سهم مدل زمانی است.
- **همه‌ی نسخه‌هایی که CLIP یا DINOv2 دارند** در epoch 1 تا 3 به بهترین حالت می‌رسند و بین ۰٫۶۸ و ۰٫۷۱ می‌مانند. تغییر pooling، conv یا طول پنجره این را عوض نمی‌کند؛ پس مشکل از ورودی است، نه از معماری.
- حذف `d_motion` و `change` وقتی ویژگی‌های ظاهری هستند اثری ندارد (۰٫۶۸۶)؛ اثرشان کنار حرکت در بخش ۸ سنجیده می‌شود.

</div>

## 8. Removing the Video Fingerprint

<div dir="rtl">

دو راه برای اینکه مدل نتواند ویدئوها را حفظ کند، و یک اصلاح برای لبه‌های ویدئو:

1. **حرکت به‌عنوان پایه (`Motion + change`):** motion و `d_motion` به‌اضافه‌ی سه عدد `change`، در کل ۱۵ بعد. `change` تنها بخش ظاهری است که اثر انگشت ویدئو نیست، چون فقط می‌گوید تصویر چقدر عوض شد، نه اینکه چه شکلی است.
2. **ویژگی ظاهری بدون اثر انگشت:**
   - **`center`:** میانگین embedding هر ویدئو، روی همه‌ی لحظه‌هایش، از هر لحظه کم می‌شود. آنچه ثابت است (نوع جاده، نور، رنگ ماشین‌ها) حذف می‌شود و فقط تغییر نسبت به حالت عادی همان ویدئو می‌ماند؛ همان چیزی که در لحظه‌ی حادثه رخ می‌دهد. این کار از کل ویدئو استفاده می‌کند و به زمان حادثه ربطی ندارد.
   - **`pca`:** ۲۰۴۸ بعد ظاهری با PCA به ۳۲ بعد کم می‌شود تا ظرفیت حفظ کردن کم شود. PCA در هر fold فقط روی لحظه‌های ویدئوهای train همان fold fit می‌شود.
   - dropout یا weight decay قوی‌تر راه دیگری بود، ولی فقط حفظ کردن را کند می‌کند و ظرفیتش سر جایش می‌ماند؛ حذف خود اثر انگشت یا کم کردن ابعاد مستقیم‌تر است.
3. **`edge_trim`:** لحظه‌های ابتدا و انتهای ویدئو الگویی دارند که وسط ویدئو نیست: conv آن‌جا به صفرهای padding می‌رسد و در لحظه‌ی اول `d_motion` و `change` دقیقاً صفرند. مدل از این الگو می‌تواند «اول ویدئو» و «آخر ویدئو» را تشخیص دهد، یعنی نوعی اطلاعات جایگاه که نمی‌خواهیم؛ در بخش ۱۰ دیده می‌شود که مدل اول واقعاً به لبه‌ها نگاه می‌کرد. با `edge_trim` فقط لحظه‌هایی در pooling شرکت می‌کنند که کل پنجره‌ی conv آن‌ها داخل ویدئوست؛ حدود ۱ ثانیه از هر سر کنار می‌رود.

همه‌ی ردیف‌ها به‌جز یکی `edge_trim` دارند؛ آن یکی اثر خود `edge_trim` را نشان می‌دهد.

</div>

In [9]:
MC = "Motion + change"
FIXES = {
    "mc_edge": ("Motion + change", {"features": MC, "model": {"edge_trim": True}}),
    "mc": ("Motion + change, no edge trim", {"features": MC, "model": {}}),
    "mc_dim64": ("Motion + change, dim 64", {"features": MC, "model": {"edge_trim": True, "dim": 64}}),
    "mc_max": ("Motion + change, max pooling", {"features": MC, "model": {"edge_trim": True, "pooling": "max"}}),
    "all_pca32": ("All, appearance PCA 32", {"features": "All", "model": {"edge_trim": True, "pca": 32}}),
    "all_center": ("All, appearance centered", {"features": "All", "model": {"edge_trim": True, "center": True}}),
    "all_center_pca32": ("All, appearance centered + PCA 32",
                         {"features": "All", "model": {"edge_trim": True, "center": True, "pca": 32}}),
}

rows = []
for key, (label, cfg) in FIXES.items():
    CONFIGS[label], OOF[label] = cfg, run_cv(key, cfg)["oof"]
    rows.append(summarize(label, OOF[label]))
fixes = pd.DataFrame(rows)
display(fixes.round(3))
metric_bars(fixes, "Without the video fingerprint (dev, 5-fold CV)")

,model,epochs,accuracy,fold_acc_std,f1,roc_auc,log_loss
0,Motion + change,13,0.723,0.020,0.731,0.800,0.565
1,"Motion + change, no edge trim",13,0.716,0.022,0.727,0.801,0.566
2,"Motion + change, dim 64",36,0.725,0.024,0.737,0.808,0.589
3,"Motion + change, max pooling",36,0.749,0.030,0.762,0.829,0.524
4,"All, appearance PCA 32",2,0.686,0.008,0.682,0.748,0.609
5,"All, appearance centered",1,0.700,0.040,0.712,0.759,0.604
6,"All, appearance centered + PCA 32",2,0.694,0.038,0.678,0.749,0.605


<div dir="rtl">

**نتیجه:**
- **centering و PCA مشکل را حل نکردند:** هر سه نسخه‌ی دارای ویژگی ظاهری بین ۰٫۶۸۶ و ۰٫۷۰۰ ماندند و باز در epoch 1 یا 2 به بهترین حالت رسیدند. ویژگی‌های ظاهری هر فریم، حتی بعد از حذف بخش ثابتشان، به تشخیص لحظه‌ی حادثه کمکی نمی‌کنند و فقط حفظ کردن را آسان می‌کنند. از اینجا به بعد مدل زمانی فقط ویژگی‌های دینامیک (حرکت و `change`) می‌بیند و ظاهر صحنه از راه linear probe در ensemble بخش ۱۱ وارد می‌شود.
- **بهترین مدل: Motion + change با max pooling:** accuracy ۰٫۷۴۹، F1 ۰٫۷۶۲ و AUC ۰٫۸۲۹. برای ویژگی‌های حرکتی max pooling از attention بهتر بود (۰٫۷۴۹ در برابر ۰٫۷۲۳): حادثه یک لحظه‌ی کوتاه است و max همان قوی‌ترین لحظه را می‌گیرد. بهترین epoch آن ۳۶ از ۴۰ بود؛ یعنی سقف ۴۰ احتمالاً کم بود و در بخش‌های بعد ۸۰ است.
- **`edge_trim`:** روی دقت تقریباً بی‌اثر بود (۰٫۷۲۳ در برابر ۰٫۷۱۶)، ولی جلوی نگاه کردن مدل به لبه‌های ویدئو را می‌گیرد (بخش ۱۲)، پس در همه‌ی مدل‌های بعدی می‌ماند.

</div>

## 9. Mirror Augmentation and Longer Training

<div dir="rtl">

همه‌ی ۹۶۰ ویدئوی train هر fold استفاده می‌شوند، ولی learning curve (بخش ۱۳) نشان می‌دهد مدل با ویدئوی بیشتر هنوز بهتر می‌شود و ویدئوی برچسب‌دار بیشتری در دیتاست نیست. دو کار بدون داده‌ی جدید:

- **mirror augmentation:** در آموزش، هر ویدئو با احتمال ۵۰٪ چپ‌وراست قرینه می‌شود. برای ویژگی‌های دینامیک یعنی فقط علامت حرکت افقی (`flow_x` و تغییرش) عوض می‌شود؛ شدت حرکت، سرعت، نزدیک شدن، تکان عمودی و `change` در تصویر قرینه همان می‌مانند. ویدئوی قرینه هنوز یک صحنه‌ی واقعی رانندگی است (مثل کشوری که از سمت چپ رانندگی می‌کند) با همان برچسب؛ پس در عمل تعداد ویدئوهای متفاوت آموزشی دو برابر می‌شود. این کار فقط برای ویژگی‌های دینامیک ممکن است؛ embedding CLIP و DINOv2 را بدون استخراج دوباره نمی‌شود قرینه کرد.
- **سقف ۸۰ epoch:** بهترین مدل بخش ۸ در epoch ۳۶ از ۴۰ بود. ردیف اول همان مدل با سقف ۸۰ است تا اثر epoch بیشتر از اثر augmentation جدا شود.

هر دو با Motion + change، max pooling و `edge_trim`.

</div>

In [10]:
MC = "Motion + change"
MAXPOOL = {"edge_trim": True, "pooling": "max"}
SPATIAL = {
    "mc_max80": ("Motion + change, 80 epochs", {"features": MC, "model": MAXPOOL, "epochs": EPOCHS_LONG}),
    "mc_max80_flip": ("Motion + change, mirror", {"features": MC, "model": MAXPOOL, "epochs": EPOCHS_LONG, "flip": True}),
}

rows = []
for key, (label, cfg) in SPATIAL.items():
    CONFIGS[label], OOF[label] = cfg, run_cv(key, cfg)["oof"]
    rows.append(summarize(label, OOF[label]))
spatial = pd.DataFrame(rows)
display(spatial.round(3))
metric_bars(spatial, "Mirror augmentation and longer training (dev, 5-fold CV)")

,model,epochs,accuracy,fold_acc_std,f1,roc_auc,log_loss
0,"Motion + change, 80 epochs",45,0.753,0.022,0.759,0.829,0.519
1,"Motion + change, mirror",63,0.760,0.020,0.763,0.834,0.508


<div dir="rtl">

**نتیجه:** سقف ۸۰ epoch کمی کمک کرد (۰٫۷۴۹ به ۰٫۷۵۳، بهترین epoch ۴۵) و mirror augmentation بیشتر: accuracy ۰٫۷۶۰، AUC ۰٫۸۳۴ و log loss ۰٫۵۰۸ در برابر ۰٫۵۱۹. با mirror بهترین epoch از ۴۵ به ۶۳ رسید؛ داده‌ی متنوع‌تر یعنی مدل دیرتر overfit می‌شود. hyperparameter search روی همین مدل انجام می‌شود.

</div>

## 10. Hyperparameter Search

<div dir="rtl">

تا اینجا hyperparameterها دستی بودند و فقط یکی‌یکی (ablation) عوض شدند. این بخش روی بهترین مدل بخش ۹ (همان ورودی و همان augmentation) یک random search انجام می‌دهد:

| hyperparameter | مقدارها |
|---|---|
| `pooling` | max، attention |
| `dim` (ابعاد نمایش هر لحظه) | 32، 64، 128 |
| `kernel` (طول پنجره‌ی conv) | 3، 5، 9 |
| `layers` (تعداد لایه‌ی conv) | 1، 2، 3 |
| `dropout` | 0.1، 0.3، 0.5 |
| `input_dropout` | 0، 0.2، 0.4 |
| `time_drop` | 0، 0.1، 0.2 |
| `lr` | 1e-4، 3e-4، 1e-3 |
| `weight_decay` | 0.01، 0.05، 0.2 |

- **random search، نه grid search:** grid کامل این جدول ۳ به توان ۹ تقریباً بیست هزار ترکیب است. random search با همان بودجه برای هر hyperparameter مقدارهای بیشتری می‌بیند، و معمولاً فقط چند hyperparameter واقعاً مهم‌اند (Bergstra و Bengio، ۲۰۱۲). Optuna (TPE) هوشمندتر انتخاب می‌کند، ولی با ۶ trial و هدف پرنوسان CV مزیتش کم است و یک وابستگی اضافه است.
- **۶ trial، هر کدام با همان 5-fold CV و ۱ seed:** بودجه‌ی GPU محدود است و با ۳ seed زمان سه برابر می‌شد. trial صفر همان تنظیمات فعلی است تا نقطه‌ی شروع هم با همین شرایط سنجیده شود.
- **معیار:** کمترین log loss out-of-fold، و تعداد epoch هر trial از منحنی خودش؛ همان قاعده‌ی بخش‌های قبل.
- **بهترین trial دوباره با ۳ seed تازه (10، 11، 12) آموزش می‌بیند.** بهترین از بین ۶ trial تک‌seed بخشی از برتری‌اش را از شانس seed گرفته است؛ seedهای تازه این بخش را حذف می‌کنند. خوش‌بینی ناشی از انتخاب روی همین foldها می‌ماند و فقط test آن را نشان می‌دهد.
- نتیجه‌ی هر trial در `mil_runs/search_XX.npz` ذخیره می‌شود؛ برای search دوباره با فضای دیگر، این فایل‌ها را پاک کنید.

</div>

In [11]:
SEARCH_BASE_LABEL = spatial.loc[spatial["log_loss"].idxmin(), "model"]
SEARCH_BASE = CONFIGS[SEARCH_BASE_LABEL]
print("Search on:", SEARCH_BASE_LABEL, SEARCH_BASE)

SPACE = {
    "pooling": ["max", "attention"],
    "dim": [32, 64, 128],
    "kernel": [3, 5, 9],
    "layers": [1, 2, 3],
    "dropout": [0.1, 0.3, 0.5],
    "input_dropout": [0.0, 0.2, 0.4],
    "time_drop": [0.0, 0.1, 0.2],
    "lr": [1e-4, 3e-4, 1e-3],
    "weight_decay": [0.01, 0.05, 0.2],
}
N_TRIALS = 6
current = {"pooling": "max", "dim": 128, "kernel": 5, "layers": 2, "dropout": 0.3, "input_dropout": 0.3,
           "time_drop": 0.1, "lr": LR, "weight_decay": WEIGHT_DECAY}
rng = np.random.default_rng(0)
trial_params = [current] + [{k: v[rng.integers(len(v))] for k, v in SPACE.items()} for _ in range(N_TRIALS - 1)]


def search_config(params):
    model = {k: v for k, v in params.items() if k not in ("lr", "weight_decay")}
    return {**SEARCH_BASE, "lr": params["lr"], "weight_decay": params["weight_decay"], "model": {"edge_trim": True, **model}}


started = time.time()
rows = []
for i, params in enumerate(tqdm(trial_params, desc="trials")):
    oof = run_cv(f"search_{i:02d}", search_config(params), seeds=[0])["oof"]
    rows.append({"trial": i, **params, **summarize(f"trial {i}", oof)})
search = pd.DataFrame(rows).drop(columns="model").sort_values("log_loss").reset_index(drop=True)
print(f"{N_TRIALS} trials in {time.time() - started:.0f} s")
display(search.head(10).round(4))

Search on: Motion + change, mirror {'features': 'Motion + change', 'model': {'edge_trim': True, 'pooling': 'max'}, 'epochs': 80, 'flip': True}


trials:   0%|          | 0/6 [00:00<?, ?it/s]

search_02:   0%|          | 0/5 [00:00<?, ?it/s]

search_03:   0%|          | 0/5 [00:00<?, ?it/s]

search_04:   0%|          | 0/5 [00:00<?, ?it/s]

search_05:   0%|          | 0/5 [00:00<?, ?it/s]

6 trials in 386 s


,trial,pooling,dim,kernel,layers,dropout,input_dropout,time_drop,lr,weight_decay,epochs,accuracy,fold_acc_std,f1,roc_auc,log_loss
0,4,max,128,9,3,0.1,0.0,0.2,0.0001,0.05,16,0.7700,0.0335,0.7596,0.8355,0.5049
1,0,max,128,5,2,0.3,0.3,0.1,0.0003,0.05,22,0.7483,0.0292,0.7541,0.8252,0.5167
2,5,max,32,5,2,0.3,0.0,0.0,0.0001,0.01,80,0.7358,0.0334,0.7421,0.8092,0.5369
3,1,attention,64,5,1,0.1,0.0,0.0,0.0001,0.01,56,0.7367,0.0233,0.7371,0.8042,0.5434
4,3,attention,128,3,3,0.5,0.0,0.1,0.0010,0.05,19,0.7533,0.0165,0.7390,0.8113,0.5686
5,2,attention,64,9,2,0.3,0.4,0.2,0.0003,0.05,32,0.7258,0.0327,0.7190,0.7981,0.5947


In [12]:
plot = search.assign(pooling=(search["pooling"] == "max").astype(int), log10_lr=np.log10(search["lr"]),
                     log10_weight_decay=np.log10(search["weight_decay"]))
dims = ["pooling", "dim", "kernel", "layers", "dropout", "input_dropout", "time_drop", "log10_lr", "log10_weight_decay", "accuracy"]
fig = px.parallel_coordinates(plot, dimensions=dims, color="accuracy", color_continuous_scale="Viridis",
                              title="Random search: one line per trial (pooling: 0 = attention, 1 = max)")
fig.update_layout(height=450)
fig.show()

TUNED_LABEL = "Tuned (random search)"
TUNED = search_config(trial_params[int(search.loc[0, "trial"])])
CONFIGS[TUNED_LABEL], OOF[TUNED_LABEL] = TUNED, run_cv("tuned", TUNED, seeds=SEEDS_FINAL)["oof"]
tuned = summarize(TUNED_LABEL, OOF[TUNED_LABEL])
print("Best trial:", TUNED)
display(pd.DataFrame([summarize(SEARCH_BASE_LABEL + " (before search)", OOF[SEARCH_BASE_LABEL]), tuned]).round(3))

tuned:   0%|          | 0/5 [00:00<?, ?it/s]

Best trial: {'features': 'Motion + change', 'model': {'edge_trim': True, 'pooling': 'max', 'dim': 128, 'kernel': 9, 'layers': 3, 'dropout': 0.1, 'input_dropout': 0.0, 'time_drop': 0.2}, 'epochs': 80, 'flip': True, 'lr': 0.0001, 'weight_decay': 0.05}


,model,epochs,accuracy,fold_acc_std,f1,roc_auc,log_loss
0,"Motion + change, mirror (before search)",63,0.760,0.020,0.763,0.834,0.508
1,Tuned (random search),21,0.772,0.023,0.761,0.846,0.487


<div dir="rtl">

**نتیجه:**
- بهترین trial (شماره‌ی ۴): max pooling، dim 128، kernel 9 با ۳ لایه، dropout ۰٫۱، بدون input dropout، time drop ۰٫۲ و lr برابر 1e-4. با kernel 9 و ۳ لایه هر لحظه ۲۵ لحظه، یعنی ۵ ثانیه، را می‌بیند. در بخش ۷ پنجره‌ی بلندتر کمکی نکرد، ولی آن‌جا ورودی ویژگی‌های ظاهری داشت و مدل در epoch 1 overfit می‌شد؛ با ورودی حرکتی، ۵ ثانیه برای دیدن قبل و بعد از حادثه (مثلاً نزدیک شدن و بعد توقف) مفید است.
- دو trial برتر هر دو max pooling هستند و trialهای attention عقب‌ترند؛ همان نتیجه‌ی بخش ۸.
- با ۳ seed تازه، مدل tuned به accuracy ۰٫۷۷۲، AUC ۰٫۸۴۶ و log loss ۰٫۴۸۷ رسید، در برابر ۰٫۷۶۰، ۰٫۸۳۴ و ۰٫۵۰۸ پیش از search. بهبود accuracy داخل بازه‌ی اطمینان است، ولی AUC و log loss هم هم‌جهت بهتر شده‌اند.

</div>

## 11. Final Model and Ensemble

<div dir="rtl">

**مدل زمانی نهایی:** از همه‌ی مدل‌های بخش‌های ۶ تا ۱۰ (به‌جز trialهای تک‌seed)، مدلی که کمترین log loss out-of-fold را دارد؛ همان قاعده‌ی انتخاب epoch.

**ensemble:** دو مدل زمانی برتر به‌اضافه‌ی linear probe. linear probe روی ویژگی‌های ظاهری میانگین‌گرفته در زمان کار می‌کند و چیزی را می‌بیند که مدل زمانی کنار گذاشته است: ظاهر کلی صحنه. احتمال نهایی میانگین وزن‌دار سه مدل است:

`p = w1 · p_final + w2 · p_second + w3 · p_probe`

وزن‌ها با گام ۰٫۰۵ (جمع ۱) با کمترین log loss out-of-fold انتخاب می‌شوند. probe با همان تنظیمات گام ۱ و همان foldها دوباره ساخته می‌شود تا برای هر ویدئو احتمال out-of-fold داشته باشیم. در نسخه‌ی قبل، ترکیب مدل حرکتی با probe accuracy را از ۰٫۷۴۹ به ۰٫۷۶۳ رساند و این سود برای وزن‌های ۰٫۴ تا ۰٫۸۵ تقریباً ثابت بود؛ یعنی سود از اطلاعات متفاوت دو مدل می‌آید، نه از تنظیم دقیق وزن. ترکیب در سطح احتمال ساده‌تر از یک مدل مشترک است و هر مدل جدا قابل بررسی می‌ماند.

چون همه‌ی این انتخاب‌ها روی همین ۱۲۰۰ ویدئو انجام شده‌اند، عدد نهایی CV کمی خوش‌بینانه است؛ عدد بی‌طرف را فقط test می‌دهد.

</div>

In [13]:
named = pd.concat([results, fixes, spatial, pd.DataFrame([tuned])], ignore_index=True)
ranked = named.sort_values("log_loss").reset_index(drop=True)
FINAL_LABEL, final_epochs = ranked.loc[0, "model"], int(ranked.loc[0, "epochs"])
SECOND_LABEL, second_epochs = ranked.loc[1, "model"], int(ranked.loc[1, "epochs"])
FINAL, SECOND = CONFIGS[FINAL_LABEL], CONFIGS[SECOND_LABEL]
p_final, p_second = OOF[FINAL_LABEL][final_epochs - 1], OOF[SECOND_LABEL][second_epochs - 1]
print(f"Final model: {FINAL_LABEL} | {FINAL} | {final_epochs} epochs")
print(f"Second model: {SECOND_LABEL} | {SECOND} | {second_epochs} epochs")


def pooled_features(row):
    """Time-pooled features of the linear probe in the preprocessing notebook."""
    s, e = offsets[row], offsets[row + 1]
    parts = []
    for key in ("clip", "dino"):
        x = cache[key][s:e].astype(np.float32)
        parts += [x.mean(0), x.max(0), x.std(0)]
    m = cache["motion"][s:e]
    parts += [m.mean(0), m.std(0), m.min(0), m.max(0), np.abs(np.diff(m, axis=0)).max(0)]
    return np.concatenate(parts)


probe_path = RUN_DIR / "probe_oof.npy"
if probe_path.exists():
    p_probe = np.load(probe_path)
else:
    P = np.stack([pooled_features(r) for r in tqdm(dev_rows)])
    probe = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
                         {"logisticregression__C": [1e-3, 1e-2, 1e-1, 1]}, cv=3, scoring="roc_auc")
    p_probe = cross_val_predict(probe, P, y_np, cv=PredefinedSplit(fold), method="predict_proba")[:, 1]
    np.save(probe_path, p_probe)

members = np.stack([p_final, p_second, p_probe])
steps = np.round(np.arange(0, 1.0001, 0.05), 2)
candidates = [np.array([a, b, round(1 - a - b, 2)]) for a in steps for b in steps if a + b <= 1.0001]
losses = np.array([log_loss(y_np, w @ members) for w in candidates])
w_best = candidates[int(np.argmin(losses))]
p_ens = w_best @ members
top10 = [accuracy_score(y_np, candidates[i] @ members >= 0.5) for i in np.argsort(losses)[:10]]
print(f"Weights: final {w_best[0]}, second {w_best[1]}, probe {w_best[2]} | "
      f"accuracy of the 10 best weight combinations: {min(top10):.3f}-{max(top10):.3f}")

ENSEMBLE_LABEL = "Ensemble: final + second + linear probe"
ensemble = metrics(ENSEMBLE_LABEL, p_ens)
final_table = pd.DataFrame([metrics(FINAL_LABEL, p_final, final_epochs), metrics(SECOND_LABEL, p_second, second_epochs),
                            metrics("Linear probe (time-pooled, all features)", p_probe), ensemble])
final_table["weight"] = list(w_best) + [np.nan]
final_table["ci_low"], final_table["ci_high"] = wilson(final_table["accuracy"], len(dev))
display(final_table.round(3))
metric_bars(final_table, "Final models and ensemble (dev, out-of-fold)")

names = ["No accident", "Accident"]
fig = px.imshow(confusion_matrix(y_np, p_ens >= 0.5), text_auto=True, x=names, y=names, color_continuous_scale="Blues",
                labels=dict(x="predicted", y="true", color="videos"), title="Confusion matrix of the ensemble (dev, out-of-fold)")
fig.update_layout(width=450, height=420)
fig.show()

Final model: Tuned (random search) | {'features': 'Motion + change', 'model': {'edge_trim': True, 'pooling': 'max', 'dim': 128, 'kernel': 9, 'layers': 3, 'dropout': 0.1, 'input_dropout': 0.0, 'time_drop': 0.2}, 'epochs': 80, 'flip': True, 'lr': 0.0001, 'weight_decay': 0.05} | 21 epochs
Second model: Motion + change, mirror | {'features': 'Motion + change', 'model': {'edge_trim': True, 'pooling': 'max'}, 'epochs': 80, 'flip': True} | 63 epochs
Weights: final 0.7, second 0.05, probe 0.25 | accuracy of the 10 best weight combinations: 0.781-0.784


,model,epochs,accuracy,fold_acc_std,f1,roc_auc,log_loss,weight,ci_low,ci_high
0,Tuned (random search),21.0,0.772,0.023,0.761,0.846,0.487,0.70,0.747,0.795
1,"Motion + change, mirror",63.0,0.760,0.020,0.763,0.834,0.508,0.05,0.735,0.783
2,"Linear probe (time-pooled, all features)",NaN,0.699,0.014,0.697,0.779,0.586,0.25,0.673,0.724
3,Ensemble: final + second + linear probe,NaN,0.781,0.020,0.773,0.859,0.468,NaN,0.757,0.803


<div dir="rtl">

**نتیجه:** ensemble به accuracy ۰٫۷۸۱ (بازه‌ی ۰٫۷۵۷ تا ۰٫۸۰۳)، F1 ۰٫۷۷۳ و AUC ۰٫۸۵۹ رسید. وزن‌ها: مدل tuned ۰٫۷۰، مدل mirror ۰٫۰۵ و linear probe ۰٫۲۵. وزن کم مدل دوم یعنی اطلاعات تازه‌ای نسبت به مدل tuned ندارد و سود ensemble از probe می‌آید که ظاهر صحنه را می‌بیند. ۱۰ ترکیب برتر وزن‌ها همه بین ۰٫۷۸۱ و ۰٫۷۸۴ هستند، پس نتیجه به وزن دقیق حساس نیست.

</div>

## 12. Where Does the Model Look?

<div dir="rtl">

مدل هیچ‌وقت زمان حادثه را ندیده است. اگر واقعاً حادثه را تشخیص می‌دهد، در ویدئوهای تصادف بیشترین وزن attention باید حوالی لحظه‌ی حادثه باشد. `time_of_event` فقط اینجا و فقط برای همین بررسی خوانده می‌شود.

مدل اول و مدل زمانی نهایی روی fold 0 (سه seed، با تعداد epoch انتخاب‌شده‌ی هر کدام) دوباره آموزش می‌بینند و روی ۲۴۰ ویدئوی validation همان fold بررسی می‌شوند. دو معیار، فقط روی ویدئوهای تصادف:
- درصدی که بیشترین وزن در فاصله‌ی ۲± ثانیه از حادثه است. اگر مدل یک لحظه‌ی تصادفی انتخاب می‌کرد، این عدد حدود ۴ ثانیه تقسیم بر طول ویدئو (حدود ۱۰٪) بود.
- درصدی که بیشترین وزن در ثانیه‌ی اول یا آخر ویدئوست (مشکل لبه‌ها، بخش ۸).

برای max pooling به‌جای وزن attention امتیاز هر لحظه (خروجی head روی همان لحظه) استفاده می‌شود. نمودار آخر وزن‌ها را برای سه ویدئوی تصادف و دو ویدئوی عادی نشان می‌دهد (اولین ویدئوهای fold، بدون انتخاب دستی)؛ خط‌چین زمان حادثه است.

</div>

In [14]:
meta = pd.read_csv(PROJECT_DIR / "metadata_all.csv", dtype={"video_id": str})
event_time = dev[["path"]].merge(meta[["path", "time_of_event"]], on="path", how="left")["time_of_event"].to_numpy()

tr, va = np.flatnonzero(fold != 0), np.flatnonzero(fold == 0)
moment_time = [cache["times"][offsets[r]:offsets[r + 1]] for r in dev_rows[va]]
positive = (y_np[va] == 1) & ~np.isnan(event_time[va])
chance = np.mean([min(4, t[-1]) / t[-1] for t, p in zip(moment_time, positive) if p])


def look(cfg, epochs):
    outputs = [predict(train(cfg, tr, seed, epochs)[0], va, with_weights=True) for seed in SEEDS]
    prob = np.mean([p for p, _ in outputs], axis=0)
    weights = [np.mean(w, axis=0) for w in zip(*[w for _, w in outputs])]
    peak = np.array([t[np.argmax(w)] for t, w in zip(moment_time, weights)])
    return prob, weights, peak


rows, looks = [], {}
for label, cfg, n_epochs in [("First model (all features)", MAIN, main_epochs), (FINAL_LABEL, FINAL, final_epochs)]:
    prob, weights, peak = looks[label] = look(cfg, n_epochs)
    error = np.abs(peak - event_time[va])[positive]
    at_edge = np.array([(p < 1) or (p > t[-1] - 1) for p, t in zip(peak, moment_time)])[positive]
    rows.append({"model": label, "fold 0 accuracy": accuracy_score(y_np[va], prob >= 0.5),
                 "peak within ±2 s of event": (error <= 2).mean(), "peak in first/last second": at_edge.mean(),
                 "median distance (s)": np.median(error)})
display(pd.DataFrame(rows).round(3))
print(f"A random moment would be within ±2 s of the event in {chance:.0%} of accident videos.")

prob, weights, peak = looks[FINAL_LABEL]
fig = px.histogram(x=(peak - event_time[va])[positive], nbins=60, color_discrete_sequence=["#d62728"],
                   title=f"Attention peak minus event time, accident videos of fold 0 ({FINAL_LABEL})")
fig.update_layout(xaxis_title="seconds", yaxis_title="videos")
fig.show()

,model,fold 0 accuracy,peak within ±2 s of event,peak in first/last second,median distance (s)
0,First model (all features),0.696,0.303,0.143,7.906
1,Tuned (random search),0.746,0.706,0.000,0.833


A random moment would be within ±2 s of the event in 11% of accident videos.


In [15]:
show = list(np.flatnonzero(y_np[va] == 1)[:3]) + list(np.flatnonzero(y_np[va] == 0)[:2])
titles = [f"{dev.video_id[va[j]]} | {'Accident' if y_np[va[j]] else 'No accident'} | predicted p = {prob[j]:.2f}"
          for j in show]
fig = make_subplots(rows=len(show), cols=1, shared_xaxes=True, subplot_titles=titles, vertical_spacing=0.05)
for i, j in enumerate(show, start=1):
    w = np.where(np.isfinite(weights[j]), weights[j], np.nan)      # trimmed edges have no score
    fig.add_trace(go.Scatter(x=moment_time[j], y=w, line_color="#d62728" if y_np[va[j]] else "#1f77b4",
                             showlegend=False), row=i, col=1)
    if y_np[va[j]] and not np.isnan(event_time[va[j]]):
        fig.add_vline(x=event_time[va[j]], line_dash="dash", line_color="black", row=i, col=1)
fig.update_xaxes(title_text="seconds", row=len(show), col=1)
fig.update_layout(height=170 * len(show), title=f"Where the final model looks ({FINAL_LABEL})")
fig.show()

<div dir="rtl">

**نتیجه:** در مدل نهایی، بیشترین امتیاز لحظه در ۷۱٪ ویدئوهای تصادف در فاصله‌ی ۲± ثانیه از حادثه است (انتخاب تصادفی: ۱۱٪)، میانه‌ی فاصله ۰٫۸ ثانیه است و هیچ ویدئویی بیشترین امتیاز را در ثانیه‌ی اول یا آخر ندارد. در مدل اول این اعداد ۳۰٪ و ۱۴٪ بودند. مدل بدون اینکه زمان حادثه را دیده باشد خود حادثه را پیدا می‌کند؛ پس دقتش از دیدن حادثه می‌آید، نه از یک میان‌بُر.

</div>

## 13. Learning Curve: Number of Training Videos

<div dir="rtl">

همه‌ی مدل‌های این نوت‌بوک با همه‌ی ۹۶۰ ویدئوی train هر fold آموزش دیده‌اند. این بخش یک آزمایش «اگر داده‌ی کمتری داشتیم» است: بهترین مدل حرکتی بخش ۸ (Motion + change با max pooling) با زیرمجموعه‌های کوچک‌تر دوباره آموزش می‌بیند تا معلوم شود (۱) نسخه‌های قبلی تیم که با ۲۴۰ تا ۴۵۰ ویدئو کار کردند چقدر از کمبود داده ضرر کردند، و (۲) آیا مدل با داده‌ی بیشتر هنوز بهتر می‌شود.

- در هر fold یک زیرمجموعه‌ی متوازن از ویدئوهای train برداشته می‌شود. زیرمجموعه‌ها تودرتو هستند (۱۲۰تایی داخل ۲۴۰تایی است) تا تفاوت فقط از تعداد بیاید. ارزیابی همیشه روی کل validation همان fold است.
- تعداد گام‌های به‌روزرسانی برای همه‌ی اندازه‌ها یکسان است (برابر همان مدل با ۹۶۰ ویدئو)؛ با تعداد epoch یکسان، مدل با داده‌ی کمتر گام کمتری برمی‌داشت و کم‌آموزش می‌ماند.
- برای کوتاه شدن زمان اجرا، ۲ seed در هر fold.

اگر منحنی validation در ۹۶۰ هنوز رو به بالا باشد، مدل «گرسنه‌ی داده» است: یعنی اگر دیتاست ویدئوی بیشتری داشت بهتر می‌شد. فاصله‌ی منحنی train و validation اندازه‌ی overfitting است.

</div>

In [16]:
SIZES = [120, 240, 480, 720, 960]
LC_LABEL = "Motion + change, max pooling"
LC, lc_epochs = CONFIGS[LC_LABEL], int(fixes.set_index("model").loc[LC_LABEL, "epochs"])
n_train = int((fold != 0).sum())
total_steps = lc_epochs * math.ceil(n_train / BATCH)
curve_path = RUN_DIR / f"learning_curve_{re.sub(r'[^a-z0-9]+', '_', LC_LABEL.lower()).strip('_')}.csv"

if curve_path.exists():
    curve = pd.read_csv(curve_path)
else:
    rows = []
    for n in tqdm(SIZES):
        oof_n, train_acc = np.zeros(len(dev)), []
        for k in range(5):
            tr_all, va_k = np.flatnonzero(fold != k), np.flatnonzero(fold == k)
            order = np.random.default_rng(k).permutation(tr_all)
            if n < len(tr_all):
                tr_k = np.r_[order[y_np[order] == 1][:n // 2], order[y_np[order] == 0][:n // 2]]
            else:
                tr_k = tr_all
            epochs_n = math.ceil(total_steps / math.ceil(len(tr_k) / BATCH))
            models = [train(LC, tr_k, seed, epochs_n)[0] for seed in SEEDS[:2]]
            oof_n[va_k] = np.mean([predict(m, va_k) for m in models], axis=0)
            train_acc.append(accuracy_score(y_np[tr_k], np.mean([predict(m, tr_k) for m in models], axis=0) >= 0.5))
        rows.append({"train_videos": n, "train_acc": np.mean(train_acc), "val_acc": accuracy_score(y_np, oof_n >= 0.5),
                     "val_auc": roc_auc_score(y_np, oof_n)})
    curve = pd.DataFrame(rows)
    curve.to_csv(curve_path, index=False)
display(curve.round(3))

,train_videos,train_acc,val_acc,val_auc
0,120,1.000,0.644,0.708
1,240,0.992,0.682,0.762
2,480,0.935,0.711,0.806
3,720,0.881,0.721,0.817
4,960,0.856,0.738,0.828


In [17]:
n = curve["train_videos"].to_numpy()
lo, hi = wilson(curve["val_acc"], len(dev))
fig = go.Figure()
fig.add_vrect(x0=240, x1=450, fillcolor="#ff7f0e", opacity=0.12, line_width=0,
              annotation_text="earlier team models", annotation_position="top left")
fig.add_trace(go.Scatter(x=np.r_[n, n[::-1]], y=np.r_[hi, lo[::-1]], fill="toself", fillcolor="#d62728", mode="lines",
                         opacity=0.15, line_width=0, name="validation accuracy 95% CI", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=n, y=curve["val_acc"], mode="lines+markers", name="validation accuracy", line_color="#d62728"))
fig.add_trace(go.Scatter(x=n, y=curve["val_auc"], mode="lines+markers", name="validation ROC-AUC", line_color="#9467bd"))
fig.add_trace(go.Scatter(x=n, y=curve["train_acc"], mode="lines+markers", name="train accuracy", line_color="#7f7f7f"))
fig.update_layout(title=f"Learning curve ({LC_LABEL}, dev, 5-fold CV)",
                  xaxis_title="training videos per fold", yaxis_title="score", height=450)
fig.show()

<div dir="rtl">

**نتیجه:** accuracy با ۱۲۰ ویدئو ۰٫۶۴۴ است و با ۹۶۰ ویدئو به ۰٫۷۳۸ می‌رسد؛ با ۲۴۰ ویدئو، یعنی اندازه‌ی کار نسخه‌های قبلی تیم، ۰٫۶۸۲ است که با عددهای آن‌ها هم‌خوان است. accuracy train از ۱٫۰ به ۰٫۸۵۶ پایین می‌آید و فاصله‌ی train و validation کم می‌شود: داده‌ی بیشتر overfitting را کم می‌کند. منحنی validation در ۹۶۰ هنوز صاف نشده؛ به همین دلیل mirror augmentation (بخش ۹) کمک کرد و مدل‌های نهایی برای test روی همه‌ی ۱۲۰۰ ویدئوی dev آموزش می‌بینند.

</div>

## 14. Comparison with All Models

<div dir="rtl">

- **مدل‌های این پروژه:** baselineهای نوت‌بوک پری‌پراسس (متادیتا، حجم فایل، طول ویدئو، linear probe)، همه‌ی نسخه‌های Attention-MIL، مدل tuned و ensemble نهایی. همه روی همان ۱۲۰۰ ویدئوی dev و همان ۵ fold.
- **نسخه‌های قبلی تیم:** fine-tune کردن MViT-v2-S، ResNet18 + LSTM، YOLOv8n + LSTM و چند encoder frozen با logistic regression. این عددها از تقسیم دیگری و روی ۳۰ تا ۳۰۰ ویدئو گزارش شده‌اند و مستقیم هم‌سنگ نیستند. برای همین کنار هر accuracy بازه‌ی اطمینان ۹۵٪ رسم شده است: با ۶۰ ویدئو بازه حدود ۲۲ درصد پهنا دارد و بیشتر آن مدل‌ها از نظر آماری از هم قابل تشخیص نیستند.
- † از `time_of_event` برای انتخاب پنجره‌ی آموزشی استفاده کرده است.

تفاوت‌های اصلی با نسخه‌های قبلی: (۱) ۹۶۰ ویدئوی آموزشی در هر fold به‌جای ۲۴۰ تا ۴۵۰؛ (۲) ارزیابی روی ۱۲۰۰ ویدئو به‌جای ۶۰؛ (۳) پوشش کامل ویدئو هر ۰٫۲ ثانیه، در حالی که مثلاً MViT هشت کلیپ حدود ۳ ثانیه‌ای از هر ویدئو می‌دید و در یک ویدئوی ۴۰ ثانیه‌ای حدود یک‌سوم ویدئو بین کلیپ‌ها دیده نمی‌شد؛ (۴) ویژگی حرکتی صریح برای شبه‌تصادف‌ها که خسارت دیده‌شدنی ندارند؛ (۵) یک مدل زمانی کوچک روی ویژگی‌های frozen، به‌جای fine-tune کردن کل شبکه روی چند صد ویدئو یا logistic regression روی ویژگی میانگین‌گرفته در زمان؛ (۶) حذف میان‌بُرها: اثر انگشت ظاهری ویدئو، لبه‌های ویدئو و جایگاه زمانی؛ (۷) بدون انتخاب checkpoint یا آستانه روی validation، و hyperparameter search با CV.

</div>

In [18]:
ours_group, team_group = "This project (1200 dev videos, 5-fold CV)", "Earlier team models (one small validation set)"

step1 = [   # from the preprocessing notebook, same folds
    {"model": "Metadata only (LogReg)", "accuracy": 0.619, "roc_auc": 0.642},
    {"model": "File size only (LogReg)", "accuracy": 0.514, "roc_auc": 0.502},
    {"model": "Video duration only (LogReg)", "accuracy": 0.527, "roc_auc": 0.530},
]
probe_csv = pd.read_csv(PROJECT_DIR / "linear_probe.csv")
probe_rows = [{"model": "Linear probe, time-pooled " + r.features, "accuracy": r.accuracy, "f1": r.f1, "roc_auc": r.roc_auc}
              for r in probe_csv.itertuples()]
ours = pd.DataFrame(step1 + probe_rows + named.to_dict("records") + [ensemble])
ours = ours[["model", "accuracy", "f1", "roc_auc"]].assign(n_eval=len(dev), group=ours_group)

teammates = pd.DataFrame([    # n_eval = number of videos the result was measured on
    ("MViT-v2-S + attention MIL (seed 42, best val epoch)", 0.717, 0.679, 0.772, 60),
    ("MViT-v2-S + MIL, 3-seed ensemble", 0.700, 0.733, 0.766, 60),
    ("MViT-v2-S + MIL, 3-seed ensemble + TTA", 0.700, 0.690, 0.761, 60),
    ("MViT-v2-S + MIL, 2-seed selected ensemble", 0.683, 0.642, 0.763, 60),
    ("MViT-v2-S + MIL, 2-seed fixed 8 epochs", 0.667, 0.660, 0.779, 60),
    ("MViT-v2-S + MIL, 2-seed fixed 8 epochs + TTA", 0.667, 0.643, 0.772, 60),
    ("MViT-v2-S + MIL, dense 10 fps, 3 seeds", 0.667, 0.660, 0.763, 60),
    ("ResNet18 + BiLSTM v3, EMA, 3 seeds + TTA", 0.683, 0.655, 0.700, 60),
    ("ResNet18 + LSTM, 3 seeds + flip TTA †", 0.667, 0.722, 0.690, 60),
    ("Sequential MViT 5 fps + causal GRU, 5 seeds", 0.617, 0.610, 0.670, 60),
    ("MobileNetV3-small + LSTM", 0.550, 0.571, 0.592, 60),
    ("YOLOv8n + LSTM + attention (from scratch)", 0.633, 0.718, 0.796, 30),
    ("Frozen ResNet18 + LogReg", 0.657, 0.653, 0.691, 300),
    ("Frozen VideoMAE V2-Base + LogReg", 0.653, 0.649, 0.719, 300),
    ("Frozen MViT-v2-S + LogReg", 0.593, 0.593, 0.650, 300),
], columns=["model", "accuracy", "f1", "roc_auc", "n_eval"])
teammates["group"] = team_group

compare = pd.concat([ours, teammates], ignore_index=True).sort_values("accuracy", ascending=False)
compare["ci_low"], compare["ci_high"] = wilson(compare["accuracy"], compare["n_eval"])
display(compare.round(3).reset_index(drop=True))

,model,accuracy,f1,roc_auc,n_eval,group,ci_low,ci_high
0,Ensemble: final + second + linear probe,0.781,0.773,0.859,1200,"This project (1200 dev videos, 5-fold CV)",0.757,0.803
1,Tuned (random search),0.772,0.761,0.846,1200,"This project (1200 dev videos, 5-fold CV)",0.747,0.795
2,"Motion + change, mirror",0.760,0.763,0.834,1200,"This project (1200 dev videos, 5-fold CV)",0.735,0.783
3,"Motion + change, 80 epochs",0.753,0.759,0.829,1200,"This project (1200 dev videos, 5-fold CV)",0.728,0.777
4,"Motion + change, max pooling",0.749,0.762,0.829,1200,"This project (1200 dev videos, 5-fold CV)",0.724,0.773
5,Motion only,0.732,0.730,0.810,1200,"This project (1200 dev videos, 5-fold CV)",0.707,0.757
6,"Motion + change, dim 64",0.725,0.737,0.808,1200,"This project (1200 dev videos, 5-fold CV)",0.699,0.750
7,Motion + change,0.723,0.731,0.800,1200,"This project (1200 dev videos, 5-fold CV)",0.697,0.748
8,"MViT-v2-S + attention MIL (seed 42, best val e...",0.717,0.679,0.772,60,Earlier team models (one small validation set),0.593,0.815
9,"Motion + change, no edge trim",0.716,0.727,0.801,1200,"This project (1200 dev videos, 5-fold CV)",0.690,0.741


In [19]:
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=["Accuracy (95% CI)", "F1", "ROC-AUC"])
for group, color in [(ours_group, "#d62728"), (team_group, "#7f7f7f")]:
    part = compare[compare["group"] == group]
    for col, metric in enumerate(["accuracy", "f1", "roc_auc"], start=1):
        error = (dict(type="data", symmetric=False, array=part["ci_high"] - part["accuracy"],
                      arrayminus=part["accuracy"] - part["ci_low"]) if metric == "accuracy" else None)
        fig.add_trace(go.Bar(x=part[metric], y=part["model"], orientation="h", name=group, marker_color=color,
                             error_x=error, legendgroup=group, showlegend=col == 1), row=1, col=col)
fig.add_vrect(x0=0.78, x1=0.80, fillcolor="#2ca02c", opacity=0.15, line_width=0, row=1, col=1,
              annotation_text="target", annotation_position="top")
fig.update_yaxes(categoryorder="array", categoryarray=compare["model"][::-1])
fig.update_xaxes(range=[0.4, 1])
fig.update_layout(title="All models, sorted by accuracy", height=24 * len(compare) + 180,
                  legend=dict(orientation="h", y=-0.04))
fig.show()

## 15. Final Evaluation on the Test Set

<div dir="rtl">

همه‌ی تصمیم‌ها تا اینجا با قاعده‌های از پیش مشخص و فقط روی dev گرفته شده‌اند: مدل نهایی، مدل دوم، تعداد epoch هر کدام و وزن‌های ensemble. این بخش ۳۰۰ ویدئوی test را برای اولین و تنها بار می‌خواند و هیچ چیزی بر اساس نتیجه‌اش عوض نمی‌شود.

- **آموزش روی همه‌ی ۱۲۰۰ ویدئوی dev:** learning curve نشان داد داده‌ی بیشتر کمک می‌کند. هر مدل زمانی با ۳ seed روی همه‌ی dev و با همان تعداد epoch انتخاب‌شده در CV آموزش می‌بیند؛ آماره‌های نرمال‌سازی هم فقط از dev. linear probe هم با همان grid روی C روی کل dev fit می‌شود.
- **ensemble با همان وزن‌های بخش ۱۱.**
- **بازه‌ی اطمینان:** با ۳۰۰ ویدئو حدود ۵± درصد است؛ اختلاف چند درصدی با عدد CV در همین بازه طبیعی است.
- عدد هر عضو ensemble هم فقط برای گزارش نشان داده می‌شود؛ انتخاب از قبل انجام شده است.

</div>

In [20]:
test_rows = np.flatnonzero(splits["fold"] == -1)
test = splits.iloc[test_rows].reset_index(drop=True)
lengths_test = cache["lengths"][test_rows]
X_test = np.zeros((len(test), lengths_test.max(), N_FEATURES), np.float16)
for j, row in enumerate(tqdm(test_rows)):
    X_test[j, :lengths_test[j]] = moment_features(row)
X_test = torch.from_numpy(X_test).to(device)
lengths_test_t = torch.as_tensor(lengths_test, device=device)
mask_test = torch.arange(X_test.shape[1], device=device) < lengths_test_t[:, None]
test_data, test_idx, y_test = (X_test, mask_test, lengths_test_t), np.arange(len(test)), test["label"].to_numpy()
print(f"Test videos: {len(test)} | labels: {np.bincount(y_test)}")

all_dev = np.arange(len(dev))
p_test = {}
for label, cfg, n_epochs in [(FINAL_LABEL, FINAL, final_epochs), (SECOND_LABEL, SECOND, second_epochs)]:
    p_test[label] = np.mean([predict(train(cfg, all_dev, seed, n_epochs)[0], test_idx, data=test_data)
                             for seed in SEEDS], axis=0)

P_dev = np.stack([pooled_features(r) for r in tqdm(dev_rows)])
P_test = np.stack([pooled_features(r) for r in tqdm(test_rows)])
probe = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
                     {"logisticregression__C": [1e-3, 1e-2, 1e-1, 1]}, cv=3, scoring="roc_auc").fit(P_dev, y_np)
p_test["Linear probe"] = probe.predict_proba(P_test)[:, 1]
p_test[ENSEMBLE_LABEL] = w_best @ np.stack([p_test[FINAL_LABEL], p_test[SECOND_LABEL], p_test["Linear probe"]])


def test_metrics(label, p):
    pred = p >= 0.5
    lo, hi = wilson(accuracy_score(y_test, pred), len(test))
    return {"model": label, "accuracy": accuracy_score(y_test, pred), "ci_low": lo, "ci_high": hi,
            "f1": f1_score(y_test, pred), "roc_auc": roc_auc_score(y_test, p)}


test_table = pd.DataFrame([test_metrics(label, p) for label, p in p_test.items()])
test_table["cv_accuracy"] = [final_table.loc[i, "accuracy"] for i in [0, 1, 2, 3]]
display(test_table.round(3))

  0%|          | 0/300 [00:00<?, ?it/s]

Test videos: 300 | labels: [151 149]


  0%|          | 0/1200 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

,model,accuracy,ci_low,ci_high,f1,roc_auc,cv_accuracy
0,Tuned (random search),0.803,0.755,0.844,0.807,0.867,0.772
1,"Motion + change, mirror",0.787,0.737,0.829,0.781,0.867,0.760
2,Linear probe,0.747,0.695,0.793,0.745,0.811,0.699
3,Ensemble: final + second + linear probe,0.810,0.762,0.850,0.811,0.877,0.781


In [21]:
fig = px.imshow(confusion_matrix(y_test, p_test[ENSEMBLE_LABEL] >= 0.5), text_auto=True, x=names, y=names,
                color_continuous_scale="Blues", labels=dict(x="predicted", y="true", color="videos"),
                title="Confusion matrix of the ensemble (test, 300 videos)")
fig.update_layout(width=450, height=420)
fig.show()

row = test_table.iloc[-1]
fig = go.Figure()
for name, value, lo, hi, color in [("CV (1200 dev videos)", final_table.iloc[-1]["accuracy"], final_table.iloc[-1]["ci_low"],
                                    final_table.iloc[-1]["ci_high"], "#1f77b4"),
                                   ("Test (300 videos)", row["accuracy"], row["ci_low"], row["ci_high"], "#d62728")]:
    fig.add_trace(go.Bar(x=[name], y=[value], marker_color=color, text=[f"{value:.3f}"], showlegend=False,
                         error_y=dict(type="data", symmetric=False, array=[hi - value], arrayminus=[value - lo])))
fig.add_hrect(y0=0.78, y1=0.80, fillcolor="#2ca02c", opacity=0.15, line_width=0, annotation_text="target")
fig.update_layout(title="Ensemble accuracy: cross-validation vs test (95% CI)", yaxis_range=[0.5, 0.9], height=420, width=550)
fig.show()

<div dir="rtl">

**نتیجه‌ی نهایی روی test (۳۰۰ ویدئو، یک بار):** ensemble به accuracy **۰٫۸۱۰** (بازه‌ی ۰٫۷۶۲ تا ۰٫۸۵۰)، F1 **۰٫۸۱۱** و AUC **۰٫۸۷۷** رسید.
- عدد test کمی بالاتر از CV (۰٫۷۸۱) است. دو دلیل دارد: مدل‌های نهایی با ۱۲۰۰ ویدئو به‌جای ۹۶۰ آموزش دیده‌اند، و ۳۰۰ ویدئو بازه‌ای حدود ۵± درصد دارد. برآورد منصفانه‌ی دقت مدل حدود ۷۸ تا ۸۱ درصد است.
- ترتیب اعضای ensemble روی test همان ترتیب CV است (مدل tuned ۰٫۸۰۳، مدل mirror ۰٫۷۸۷، linear probe ۰٫۷۴۷)؛ یعنی انتخاب‌هایی که روی dev گرفته شد به داده‌ی دیده‌نشده منتقل شده‌اند.

</div>

## 16. Saved Outputs and Next Step

<div dir="rtl">

در پوشه‌ی `accident_detection/mil_runs` روی Drive:
- `<model>.npz` و `search_XX.npz`: احتمال out-of-fold هر ویدئو بعد از هر epoch (و منحنی‌های train برای مدل اول)
- `cv_results.csv`، `hyperparameter_search.csv`، `learning_curve_<model>.csv`، `all_models.csv`، `probe_oof.npy`
- `oof_final.csv`: احتمال out-of-fold مدل نهایی، مدل دوم، probe و ensemble برای هر ویدئوی dev
- `test_predictions.csv` و `test_results.csv`: احتمال و نتیجه‌ی هر مدل روی test
- `config.json`: تنظیمات دو مدل زمانی و وزن‌های ensemble

**قدم بعدی:** همه‌ی نتایج نشان دادند سیگنال اصلی در حرکت است، ولی ویژگی حرکتی فعلی میانگین کل تصویر است. نوت‌بوک Spatial Motion Features حرکت را با RAFT در یک شبکه‌ی ۳×۳ از تصویر حساب می‌کند (کجای تصویر حرکت هست). استخراج آن برای ۱۵۰۰ ویدئو روی Colab چند ساعت طول می‌کشد، چون خواندن ویدئوها روی CPU گلوگاه است، و در زمان این پروژه انجام نشد.

</div>

In [22]:
named.to_csv(RUN_DIR / "cv_results.csv", index=False)
search.to_csv(RUN_DIR / "hyperparameter_search.csv", index=False)
compare.to_csv(RUN_DIR / "all_models.csv", index=False)
test_table.to_csv(RUN_DIR / "test_results.csv", index=False)
pd.DataFrame({"video_id": dev["video_id"], "fold": fold, "label": y_np, "p_final": p_final, "p_second": p_second,
              "p_probe": p_probe, "p_ensemble": p_ens}).to_csv(RUN_DIR / "oof_final.csv", index=False)
pd.DataFrame({"video_id": test["video_id"], "label": y_test, **{k: v for k, v in p_test.items()}}).to_csv(
    RUN_DIR / "test_predictions.csv", index=False)
with open(RUN_DIR / "config.json", "w") as f:
    json.dump({"final": {"label": FINAL_LABEL, "config": FINAL, "epochs": final_epochs},
               "second": {"label": SECOND_LABEL, "config": SECOND, "epochs": second_epochs},
               "ensemble_weights": [float(w) for w in w_best], "defaults": {"lr": LR, "weight_decay": WEIGHT_DECAY,
               "batch": BATCH}, "seeds": SEEDS}, f, indent=2)
print(sorted(p.name for p in RUN_DIR.iterdir()))

['all_center.npz', 'all_center_pca32.npz', 'all_models.csv', 'all_pca32.npz', 'clip.npz', 'config.json', 'cv_results.csv', 'dino.npz', 'hyperparameter_search.csv', 'kernel9.npz', 'learning_curve.csv', 'learning_curve_motion_change_max_pooling.csv', 'main.npz', 'max_pool.npz', 'mc.npz', 'mc_dim64.npz', 'mc_edge.npz', 'mc_max.npz', 'mc_max80.npz', 'mc_max80_flip.npz', 'mean_pool.npz', 'motion.npz', 'no_change.npz', 'no_conv.npz', 'oof_final.csv', 'oof_main.csv', 'probe_oof.npy', 'search_00.npz', 'search_01.npz', 'search_02.npz', 'search_03.npz', 'search_04.npz', 'search_05.npz', 'test_predictions.csv', 'test_results.csv', 'tuned.npz']
